# El modelo *Integrate and Fire*

## TeorÃ­a

El modelo *Integrate and Fire* del comportamiento de una neurona viene descripto por dos ingredientes. En primer lugar, la ODE

\begin{eqnarray}
\frac{dV}{dt}
&=&
f(V(t))
\\
&=&
\frac{1}{\tau}
\left(
E-V(t)
+
RI(t)
\right)
\end{eqnarray}

donde:

* $V(t)$ representa el potencial de membrana al tiempo $t$,

* $E\approx -65mV$ representa el potencial de reposo,

* $\tau\approx 10ms$ representa el tiempo de relajaciÃ³n,

* $R\approx 10M\Omega$ representa una resistencia elÃ©ctrica, y

* $I(t)\approx 1nA$ representa una corriente/seÃ±al de entrada.

En segundo lugar, un mecanismo de disparo definido por

\begin{eqnarray}
V(t) \to E
\end{eqnarray}

cuando

\begin{eqnarray}
V(t) > V_u
\end{eqnarray}

donde:

* $V_u \approx -50mV$ representa un potencial umbral.

Es decir, segÃºn el mecanismo de disparo, el potencial de membrana $V(t)$ se resetea a su valor de reposo $E$ si supera cierto valor umbral $V_u$.

## Unidades

* p = pico = $10^{-12}$
* n = nano = $10^{-9}$
* $\mu$ = micro = $10^{-6}$
* m = mili = $10^{-3}$
* ...
* k = kilo = $10^{3}$
* M = Mega = $10^{6}$
* G = Giga = $10^{9}$
* T = Tera = $10^{12}$

### Ley de Ohm

Si $\Omega$ = Ohmio, V = Voltio y A = Amper, entonces

$$ 1\, Ohm = 1 \Omega = \frac{1 V}{1 A} $$

### Ley RC

Si s = segundos y F = Faradios, entonces

$$ 1 s = 1 \Omega \times 1 F $$

### Hertz

Si Hz = Hertz, luego

$$ 1 Hz = \frac{1}{1 s} $$

### Referencias

* https://technologystudent.com/elec1/ohm1.htm

* https://en.wikipedia.org/wiki/Biological_neuron_model#Leaky_integrate-and-fire

* https://compneuro.neuromatch.io/tutorials/W2D3_BiologicalNeuronModels/student/W2D3_Tutorial1.html



## **Ejercicio 1)** Importando librerÃ­as

**1)** Importe las librerÃ­as `numpy` para operar con arrays, `scipy` para utilizar rutinas de algebra lineal y `matplotlib.pyplot` para graficar.

**2)** Suba el archivo "libreria" `ode.py` que fabricÃ³ en el prÃ¡ctico 4), e impÃ³rtelo.

In [ ]:
# 1.1)
import numpy as np
import scipy as sp
import scipy.linalg as linalg
import matplotlib.pyplot as plt

In [ ]:
# 1.2)
#import ode

def euler(f,x,t,h,p):
    """
    Calcula un paso de integraciÃ³n del mÃ©todo de Euler.

    Argumentos de entrada:

        f : R^n,R -> R^n
        x = x(t) : R^n
        t = tiempo : R
        h = paso de tiempo : R
        p = parametros : R^q

    Retorna aproximacion numÃ©rica de

        x(t+h) : R^n

    segÃºn el mÃ©todo de Euler.

    # Ejemplos:
    """
    return x+h*f(x,t,p)

def rk4(f,x,t,h,p):
    """
    Calcula un paso de integraciÃ³n del mÃ©todo de Runge Kutta orden 4.

    Argumentos de entrada:

        f : R^n -> R^n
        x = x(t) : R^n
        t = tiempo : R
        h = paso de tiempo : R
        p = parametros : R^q

    Retorna aproximacion numÃ©rica de

        x(t+h) : R^n

    segÃºn el mÃ©todo de Runge-Kutta de orden 4.

    # Ejemplos:
    """
    k1 = f(x,t,p)
    k2 = f(x+0.5*h*k1,t+0.5*h,p)
    k3 = f(x+0.5*h*k2,t+0.5*h,p)
    k4 = f(x+h*k3,t+h,p)
    return x+h*(k1+2.0*k2+2.0*k3+k4)/6.0

def integrador_ode(m,f,xa,a,b,k,p,c=lambda x,t,p:x):
    """
    Integra numÃ©ricamente la ODE

        dx/dt = f(x,t)

    sobre el intervalo t:[a,b] usando k pasos de integraciÃ³n y el mÃ©todo m, bajo condicion inicial x(a)=x0.
    No es necesario que a<b.

    Argumentos de entrada:

        m = metodo de integracion (ej. euler, rk2, etc.)
        f : R^n -> R^n
        xa = condicion inicial : R
        a = tiempo inicial : R
        b = tiempo final : R
        k = num. pasos de integracion : N
        p = parametros : R^q
        c = funciÃ³n condicionante : R^n,R,p -> R^n

    Retorna:

        t : R^{k+1} , t_j = a+j*h para j=0,1,...,k
        w : R^{n,k+1} , w_ij = x_i(t_j) para i=0,1,...,n-1 y j=0,1,...,k

    donde a+k*dt = b.
    """
    assert k>0
    n = len(xa)
    h = (b-a)/k
    w = np.zeros((n,k+1)) # Produce un array con forma y tipo especificada con los parametros,
                          # lleno de ceros. la forma puede ser espcificada con un entero o tupla (n,k+1)
    t = np.zeros(k+1)
    w[:,0] = xa           # actualiza la posicion inicial (columna de indice 0) de las variables con los valores
                          # de las condiciones iniciales
    t[0] = a              # actualiza la posicion cero con el valor del tiempo inicial

    for j in range(k):    #Aca se produce la iteraciÃ³n en j

        t[j+1] = t[j] + h                # iteracion tiempo
        w[:,j+1] = m(f,w[:,j],t[j],h,p)  # iteracion de w
        w[:,j+1] = c(w[:,j+1],t[j+1],p)  # condicion sobre w

    return t,w

## **Ejercicio 2)** DinÃ¡mica con mecanismo de disparo desactivado

En lo que sigue, se considera desactivado el mecanismo de disparo.

**1)** Resuelva analÃ­ticamente la ODE del modelo *Integrate and Fire* para una condiciÃ³n inicial arbitraria $V(0)=V_0$ y una corriente de entrada $I=0$.

**2)** Grafique la soluciÃ³n para $V_0=10mV$ y $t\in [0,100]ms$. Incorpore al grÃ¡fico una lÃ­nea punteada indicando el potencial de reposo.

**3)** Realice un anÃ¡lisis geomÃ©trico de la soluciÃ³n calculada. Incorpore flechas del campo vectorial asÃ­ como los puntos fijos estables, inestables y marginales, si los hubiere.

**4)** Repita **1)**, **2)** y **3)** para el caso de corriente de entrada constante, $I=2$nA.

### Extra

**5)** Resuelva analÃ­ticamente la ODE para una corriente de entrada $I:t\to I(t)$ arbitraria.

In [ ]:
# 2.2, 2.3 y 2.4) Dinamica con mecanismo de disparo desactivado
 # Constantes fisiologicas
 E = -65.0      # mV (potencial de reposo)
 tau = 10.0     # ms (tiempo de relajacion)
 R = 10.0       # MOhm (resistencia de membrana)
 # Nota sobre unidades: R [MOhm] * I [nA] = [mV]
 `n # Solucion analitica
 def V_analitica(t, V0, E, tau, R, I):
     V_star = E + R * I
     return V_star + (V0 - V_star) * np.exp(-t / tau)
 `n t_vals = np.linspace(0, 100, 500)
 `n # Graficos 2.2 y 2.4 (Series temporales)
 fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 `n # Caso I = 0, V0 = 10 mV
 V_I0 = V_analitica(t_vals, V0=10.0, E=E, tau=tau, R=R, I=0.0)
 axes[0].plot(t_vals, V_I0, label="$V(t)$ analitica", color="darkblue", lw=2)
 axes[0].axhline(E, color="red", linestyle="--", label=f"Potencial de reposo $E={E}$ mV")
 axes[0].set_title("Ejercicio 2.2: Respuesta sin corriente ($I=0$ nA, $V_0=10$ mV)")
 axes[0].set_xlabel("Tiempo $t$ [ms]")
 axes[0].set_ylabel("Potencial de membrana $V$ [mV]")
 axes[0].legend()
 axes[0].grid(True, alpha=0.3)
 `n # Caso I = 2 nA constante, V0 = 10 mV
 I_const = 2.0
 V_star_I2 = E + R * I_const  # -65 + 20 = -45 mV
 V_I2 = V_analitica(t_vals, V0=10.0, E=E, tau=tau, R=R, I=I_const)
 axes[1].plot(t_vals, V_I2, label="$V(t)$ analitica", color="darkgreen", lw=2)
 axes[1].axhline(E, color="red", linestyle=":", label=f"Reposo $E={E}$ mV")
 axes[1].axhline(V_star_I2, color="darkorange", linestyle="--", label=f"Punto fijo $V^*={V_star_I2}$ mV")
 axes[1].set_title("Ejercicio 2.4: Respuesta con corriente constante ($I=2$ nA)")
 axes[1].set_xlabel("Tiempo $t$ [ms]")
 axes[1].set_ylabel("Potencial de membrana $V$ [mV]")
 axes[1].legend()
 axes[1].grid(True, alpha=0.3)
 `n plt.tight_layout()
 plt.show()
 `n # 2.3) Analisis geometrico y campo vectorial en el espacio de fases unidimensional (dV/dt vs V)
 fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 V_grid = np.linspace(-80, 0, 200)
 `n # Campo para I = 0
 dV_I0 = (E - V_grid) / tau
 axes[0].plot(V_grid, dV_I0, color="blue", lw=2, label=r"$\dot{V} = (E - V)/\tau$")
 axes[0].axhline(0, color="black", lw=1)
 axes[0].scatter([E], [0], color="green", s=100, zorder=5, label=f"Punto fijo estable $V^*={E}$ mV")
 axes[0].quiver(V_grid[::15], np.zeros_like(V_grid[::15]), dV_I0[::15], np.zeros_like(V_grid[::15]), scale=25, color="gray", width=0.005)
 axes[0].set_title("Campo vectorial 1D: $I = 0$ nA")
 axes[0].set_xlabel("$V$ [mV]")
 axes[0].set_ylabel(r"$dV/dt$ [mV/ms]")
 axes[0].legend()
 axes[0].grid(True, alpha=0.3)
 `n # Campo para I = 2 nA
 dV_I2 = (E - V_grid + R * I_const) / tau
 axes[1].plot(V_grid, dV_I2, color="green", lw=2, label=r"$\dot{V} = (E - V + RI)/\tau$")
 axes[1].axhline(0, color="black", lw=1)
 axes[1].scatter([V_star_I2], [0], color="green", s=100, zorder=5, label=f"Punto fijo estable $V^*={V_star_I2}$ mV")
 axes[1].quiver(V_grid[::15], np.zeros_like(V_grid[::15]), dV_I2[::15], np.zeros_like(V_grid[::15]), scale=25, color="gray", width=0.005)
 axes[1].set_title("Campo vectorial 1D: $I = 2$ nA")
 axes[1].set_xlabel("$V$ [mV]")
 axes[1].set_ylabel(r"$dV/dt$ [mV/ms]")
 axes[1].legend()
 axes[1].grid(True, alpha=0.3)
 `n plt.tight_layout()
 plt.show()


### **Resolucion analitica del Ejercicio 2**
 `n #### **2.1) Caso corriente nula ($I=0$)**
 Partiendo de la ODE lineal de primer orden:
 $$\frac{dV}{dt} = \frac{1}{\tau} (E - V(t))$$
 Separando variables:
 $$\frac{dV}{V - E} = -\frac{dt}{\tau}$$
 Integrando desde $t=0$ con $V(0)=V_0$:
 $$\int_{V_0}^{V(t)} \frac{dV'}{V' - E} = -\frac{1}{\tau} \int_0^t dt' \implies \ln\left(\frac{V(t)-E}{V_0-E}\right) = -\frac{t}{\tau}$$
 Despejando el potencial de membrana:
 $$V(t) = E + (V_0 - E) e^{-t/\tau}$$
 `n #### **2.3) Analisis Geometrico de Estabilidad**
 La derivada respecto a $V$ del campo vectorial es:
 $$\frac{d}{dV}f(V) = -\frac{1}{\tau} < 0$$
 Como la derivada es estrictamente negativa en todo el dominio, el punto fijo $V^* = E$ es **globalmente y asintoticamente estable** (un atractor lineal). Para cualquier condicion inicial $V_0$, $V(t) \to E$ exponencialmente cuando $t \to \infty$.
 `n #### **2.4) Caso corriente constante ($I = 2\text{ nA}$)**
 El nuevo punto de equilibrio $V^*$ ocurre cuando $\dot{V} = 0$:
 $$E - V^* + R I = 0 \implies V^* = E + R I = -65\text{ mV} + (10\text{ M}\Omega)(2\text{ nA}) = -65\text{ mV} + 20\text{ mV} = -45\text{ mV}$$
 Definiendo $u(t) = V(t) - V^*$, la ecuacion se transforma en $\dot{u} = -u/\tau$, cuya solucion es:
 $$V(t) = V^* + (V_0 - V^*) e^{-t/\tau} = (E + RI) + (V_0 - E - RI) e^{-t/\tau}$$
 Nuevamente, dado que $\frac{d}{dV}f(V) = -1/\tau < 0$, el nuevo punto fijo $V^* = -45\text{ mV}$ es **asintoticamente estable**.
 `n #### **2.5) Extra: Solucion general para $I(t)$ arbitraria**
 Multiplicando la ODE $\dot{V} + \frac{1}{\tau}V = \frac{1}{\tau}(E + R I(t))$ por el factor integrante $\mu(t) = e^{t/\tau}$:
 $$\frac{d}{dt}\left(V(t) e^{t/\tau}\right) = \frac{1}{\tau} (E + R I(t)) e^{t/\tau}$$
 Integrando entre $0$ y $t$:
 $$V(t) = E + (V_0 - E) e^{-t/\tau} + \frac{R}{\tau} \int_0^t e^{-(t-s)/\tau} I(s)\, ds$$
 El potencial es una convolucion causal de la corriente de entrada con el filtro exponencial de relajacion de la membrana.

## **Ejercicio 3)** DinÃ¡mica con mecanismo de disparo activado

**1)** Resuelva numÃ©ricamente la ODE del modelo *Integrate and Fire* en el intervalo $t\in [0,200]ms$ para la condiciÃ³n inicial $V(0)=E$ y una corriente de entrada $I(t)=I_0$ para todo $t$ con $I_0=2nA$. Utilice el paso de integraciÃ³n $h=0.05ms$.

**2)** Grafique la soluciÃ³n computada en el inciso **3.1)**. Incorpore al grÃ¡fico la soluciÃ³n exacta del inciso **2.1)** (en donde el mecanismo de disparo estÃ¡ desactivado) para la condiciÃ³n inicial del inciso anterior. AdemÃ¡s, grafique lÃ­neas punteadas horizontales marcando los valores de $V^*$ y $V_u$, donde $V^*$ es el valor de $V$ tal que $f(V)=0$.

**3)** Para el caso indicado en el inciso **3.1)**, calcule analÃ­ticamente el perÃ­odo de disparo. Observe si el valor computado corresponde con el observado en el inciso **3.2)**. GrafÃ­que la frencuencia en funciÃ³n de $I_0$.
Que ocurre para valores pequeÃ±os de $I_0$?

**4)** Repita **3.1)** y **3.2)** con $I_e(t)=I_0 \cos(t/30ms)$ e $I_0=2.5nA$ para $0\,ms\leq t\leq 500\,ms$. En este caso, no incluya la soluciÃ³n exacta del inciso **2.1)**, pero sÃ­ lineas punteadas horizontales para marcar los valores de $E$ y $V_u$.

**5)** Repita **3.1)** y **3.2)** con

\begin{equation}
I(t)= I_0 \, \left(
    \cos{\left( \frac{t}{3ms} \right)} +
    \sin{\left( \frac{t}{5ms} \right)} +
    \cos{\left( \frac{t}{7ms} \right)} +
    \sin{\left( \frac{t}{11ms} \right)} +
    \cos{\left( \frac{t}{13ms} \right)}  \right)^2 \, nA \,
    \;\;\;(4)
\end{equation}

e $I_0=0.35 nA$ para $0\,ms\leq t\leq 200\,ms$. Al igual que en inciso anterior, no incluya la soluciÃ³n exacta del inciso **2.a)**, pero sÃ­ lineas punteadas horizontales para marcar los valores de $E$ y $V_u$.

In [ ]:
# 3.1 a 3.5) Dinamica con mecanismo de disparo activado
 Vu = -50.0  # mV (umbral de disparo)
 `n # Ecuacion diferencial del modelo IAF
 def f_iaf(v, t, p):
     I_t = p["I_func"](t)
     return np.array([(p["E"] - v[0] + p["R"] * I_t) / p["tau"]])
 `n # Condicionante de disparo: si V > Vu, resetea a E
 def condicion_disparo(v, t, p):
     if v[0] > p["Vu"]:
         return np.array([p["E"]])
     return v
 `n # --- 3.1 y 3.2: Corriente constante I0 = 2 nA ---
 p_const = {
     "E": E, "tau": tau, "R": R, "Vu": Vu,
     "I_func": lambda t: 2.0
 }
 a_3 = 0.0
 b_3 = 200.0
 h_3 = 0.05
 k_3 = int((b_3 - a_3) / h_3)
 xa_3 = np.array([E])
 `n t_3, w_3 = integrador_ode(euler, f_iaf, xa_3, a_3, b_3, k_3, p_const, c=condicion_disparo)
 `n # Solucion analitica sin mecanismo de disparo
 V_sin_disparo = V_analitica(t_3, V0=E, E=E, tau=tau, R=R, I=2.0)
 `n plt.figure(figsize=(12, 5))
 plt.plot(t_3, w_3[0, :], label="Numerica con disparo (Euler, h=0.05 ms)", color="blue", lw=1.5)
 plt.plot(t_3, V_sin_disparo, label="Exacta sin disparo ($I=2$ nA)", color="green", linestyle="--", lw=1.5)
 plt.axhline(Vu, color="red", linestyle="--", label=f"Umbral $V_u = {Vu}$ mV")
 plt.axhline(V_star_I2, color="darkorange", linestyle=":", label=f"Asintota $V^* = {V_star_I2}$ mV")
 plt.axhline(E, color="gray", linestyle=":", label=f"Reposo $E = {E}$ mV")
 plt.title("Ejercicio 3.2: Modelo Integrate-and-Fire con corriente constante ($I_0 = 2$ nA)")
 plt.xlabel("Tiempo $t$ [ms]")
 plt.ylabel("Potencial $V(t)$ [mV]")
 plt.legend(loc="upper right")
 plt.grid(True, alpha=0.3)
 plt.show()
 `n # --- 3.3: Periodo y frecuencia analitica vs corriente I0 ---
 # Analiticamente: T = tau * ln( (RI) / (RI - (Vu - E)) ) para RI > (Vu - E)
 T_teorico = tau * np.log((R * 2.0) / (R * 2.0 - (Vu - E)))
 f_teorica = 1000.0 / T_teorico  # en Hz (1000 ms / T[ms])
 print(f"Periodo teorico T = {T_teorico:.2f} ms | Frecuencia teorica = {f_teorica:.2f} Hz")
 `n # Curva F-I (Frecuencia vs Corriente de entrada)
 I_range = np.linspace(0.5, 5.0, 300)
 I_th = (Vu - E) / R  # Corriente reobase = 15 mV / 10 MOhm = 1.5 nA
 freq_vals = np.zeros_like(I_range)
 for idx, i_val in enumerate(I_range):
     if i_val > I_th:
         T_val = tau * np.log((R * i_val) / (R * i_val - (Vu - E)))
         freq_vals[idx] = 1000.0 / T_val
     else:
         freq_vals[idx] = 0.0
 `n plt.figure(figsize=(8, 4.5))
 plt.plot(I_range, freq_vals, color="purple", lw=2)
 plt.axvline(I_th, color="red", linestyle="--", label=f"Reobase $I_{{th}} = {I_th}$ nA")
 plt.scatter([2.0], [f_teorica], color="blue", zorder=5, label=f"$I_0=2$ nA: f={f_teorica:.1f} Hz")
 plt.title("Curva F-I: Frecuencia de disparo vs Corriente de entrada")
 plt.xlabel("Corriente $I_0$ [nA]")
 plt.ylabel("Frecuencia [Hz]")
 plt.legend()
 plt.grid(True, alpha=0.3)
 plt.show()
 `n # --- 3.4: Corriente oscilante I(t) = I0 * cos(t / 30 ms) con I0 = 2.5 nA ---
 p_cos = {
     "E": E, "tau": tau, "R": R, "Vu": Vu,
     "I_func": lambda t: 2.5 * np.cos(t / 30.0)
 }
 b_cos = 500.0
 k_cos = int(b_cos / h_3)
 t_cos, w_cos = integrador_ode(euler, f_iaf, xa_3, 0.0, b_cos, k_cos, p_cos, c=condicion_disparo)
 `n fig, (ax_v, ax_i) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
 ax_v.plot(t_cos, w_cos[0, :], color="blue", lw=1.2, label="$V(t)$")
 ax_v.axhline(Vu, color="red", linestyle="--", label="$V_u = -50$ mV")
 ax_v.axhline(E, color="gray", linestyle=":", label="$E = -65$ mV")
 ax_v.set_ylabel("$V(t)$ [mV]")
 ax_v.set_title("Ejercicio 3.4: Disparo con estimulo sinusoidal $I(t) = 2.5\\cos(t/30)$ nA")
 ax_v.legend()
 ax_v.grid(True, alpha=0.3)
 `n ax_i.plot(t_cos, 2.5 * np.cos(t_cos / 30.0), color="crimson", lw=1.5, label="$I(t)$")
 ax_i.axhline(I_th, color="orange", linestyle="--", label=f"Reobase $I_{{th}}={I_th}$ nA")
 ax_i.set_xlabel("Tiempo $t$ [ms]")
 ax_i.set_ylabel("Corriente [nA]")
 ax_i.legend()
 ax_i.grid(True, alpha=0.3)
 plt.tight_layout()
 plt.show()
 `n # --- 3.5: Corriente multiharmonica compleja (Ec. 4) ---
 def I_multi(t, I0=0.35):
     suma = (np.cos(t / 3.0) + np.sin(t / 5.0) + np.cos(t / 7.0) +
             np.sin(t / 11.0) + np.cos(t / 13.0))
     return I0 * (suma ** 2)
 `n p_multi = {
     "E": E, "tau": tau, "R": R, "Vu": Vu,
     "I_func": lambda t: I_multi(t, 0.35)
 }
 b_multi = 200.0
 k_multi = int(b_multi / h_3)
 t_multi, w_multi = integrador_ode(euler, f_iaf, xa_3, 0.0, b_multi, k_multi, p_multi, c=condicion_disparo)
 `n fig, (ax_v, ax_i) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
 ax_v.plot(t_multi, w_multi[0, :], color="blue", lw=1.2, label="$V(t)$")
 ax_v.axhline(Vu, color="red", linestyle="--", label="$V_u = -50$ mV")
 ax_v.axhline(E, color="gray", linestyle=":", label="$E = -65$ mV")
 ax_v.set_ylabel("$V(t)$ [mV]")
 ax_v.set_title("Ejercicio 3.5: Disparos con corriente multiharmonica quasi-periodica")
 ax_v.legend()
 ax_v.grid(True, alpha=0.3)
 `n ax_i.plot(t_multi, I_multi(t_multi, 0.35), color="darkmagenta", lw=1.5, label="$I(t)$ multiharmonica")
 ax_i.axhline(I_th, color="orange", linestyle="--", label=f"Reobase $I_{{th}}={I_th}$ nA")
 ax_i.set_xlabel("Tiempo $t$ [ms]")
 ax_i.set_ylabel("Corriente [nA]")
 ax_i.legend()
 ax_i.grid(True, alpha=0.3)
 plt.tight_layout()
 plt.show()
